In [19]:
import pandas as pd
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error , mean_squared_error , root_mean_squared_error, r2_score

In [ ]:
df=pd.read_csv(r"C:\Users\nice\Desktop\Big Data project\Mobile-Price-prediction\Data\proccessed\processed_data.csv")
df.head(1)

,brand,price_inr,expandable_storage,gpu_score,cpu_score,build_material,wifi_version,chipset,ram_gb,display_type,refresh_rate_hz,battery_mah,rear_camera_mp,camera_setup,thickness_mm
0,Oppo,12672.0,0.0,1308.0,1683.0,Plastic,WiFi 5,Snapdragon 680,4.0,LCD,60.0,4866.0,50.0,"Macro, Wide",7.63


In [13]:
df=df.drop(columns=["thickness_mm","build_material","wifi_version","camera_setup"],axis=1)

In [14]:
df

,brand,price_inr,expandable_storage,gpu_score,cpu_score,chipset,ram_gb,display_type,refresh_rate_hz,battery_mah,rear_camera_mp
0,Oppo,12672.0,0.0,1308.0,1683.0,Snapdragon 680,4.0,LCD,60.0,4866.0,50.0
1,Samsung,63088.0,0.0,6481.0,8367.0,Tensor G3,8.0,AMOLED,120.0,4962.0,50.0
2,Motorola,19912.0,1.0,2218.0,3277.0,Snapdragon 680,3.0,OLED,60.0,5165.0,50.0
3,Apple,108140.0,0.0,10793.0,13132.0,A17 Bionic,16.0,OLED,120.0,3512.0,108.0
4,Apple,93543.0,0.0,8283.0,12713.0,A17 Bionic,12.0,OLED,90.0,3731.0,200.0
...,...,...,...,...,...,...,...,...,...,...,...
178830,Oppo,29803.0,0.0,2624.0,3579.0,Dimensity 900,12.0,AMOLED,144.0,4456.0,50.0
178831,Oppo,28203.0,0.0,3080.0,3808.0,Exynos 1380,6.0,AMOLED,90.0,7992.0,64.0
178832,Apple,102164.0,0.0,11624.0,12356.0,Snapdragon 8 Gen 2,12.0,OLED,90.0,3296.0,50.0
178833,Google,55333.0,0.0,5397.0,6305.0,Snapdragon 8 Gen 2,16.0,AMOLED,120.0,4830.0,50.0


In [21]:
cat=["brand", "chipset", "display_type"]
processesor = ColumnTransformer(
    transformers=[
        ("cat_process", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat)
    ], 
    remainder="passthrough"
)

In [22]:
full_pipeline = Pipeline(steps=[
    ('preprocessor', processesor),
    ('regressor', XGBRegressor(early_stopping_rounds=10))
])

In [23]:
x=df.drop(columns=["price_inr"])
y=df["price_inr"]
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.3, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

In [24]:
full_pipeline.named_steps['preprocessor'].fit(x_train)
x_val_transformed=full_pipeline.named_steps['preprocessor'].transform(x_val)
full_pipeline.fit(x_train,y_train,regressor__eval_set=[(x_val_transformed, y_val)],regressor__verbose=False)
y_pred = full_pipeline.predict(x_test)
print("MAE=",mean_absolute_error(y_test,y_pred))
print("MSE=",mean_squared_error(y_test,y_pred))
print("RSME=",root_mean_squared_error(y_test,y_pred))
print("R2_Score=",r2_score(y_test,y_pred))


MAE= 2167.526259278181
MSE= 7562922.867080039
RSME= 2750.0768838488934
R2_Score= 0.9886614111408646
